In [1]:
import os
import pandas as pd
import ast
from Bio import SeqIO
import warnings

warnings.filterwarnings('ignore')

def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def coverage_calcu(align_result, label, len_query):
    acc_count = align_result['sseqid'].value_counts()
    acc_list = list(acc_count.index)

    rest_result = align_result
    rest_result['sseqid'] = pd.Categorical(rest_result['sseqid'], categories=acc_list, ordered=True)
    rest_result = rest_result.sort_values(by = 'sseqid').reset_index(drop=True)
    
    coverage_pd = pd.DataFrame()
    row_start = 0
    row_end = 0
    for sub_acc in acc_count.index:
        row_start = row_end
        row_end += acc_count[sub_acc]
        temp_pd = rest_result.iloc[row_start: row_end][['qstart', 'qend']]
        
        start_pd = pd.concat([temp_pd[['qstart']].sort_values(by='qstart'), pd.DataFrame([{'qstart': len_query+1}])], ignore_index=True)
        end_pd = pd.concat([pd.DataFrame([{'qend': 0}]), temp_pd[['qend']].sort_values(by='qend')], ignore_index=True)
        new_pd = pd.concat([start_pd, end_pd], axis=1)
        new_pd = new_pd[new_pd['qstart'] > new_pd['qend']].reset_index(drop=True)
        start_pd = new_pd.iloc[0:len(new_pd)-1][['qstart']].reset_index(drop=True)
        end_pd = new_pd.iloc[1:len(new_pd)][['qend']].reset_index(drop=True)
        temp_list_pd = pd.concat([start_pd, end_pd], axis=1)
        
        temp_list_pd['length'] = temp_list_pd['qend'] - temp_list_pd['qstart'] + 1
        temp_list_pd['range'] = temp_list_pd['qstart'].astype(str) + '-' + temp_list_pd['qend'].astype(str)
        sum_len = sum(temp_list_pd['length'])
        
        coverage_pd = pd.concat([coverage_pd, pd.DataFrame([{'id': sub_acc, f'coverage-{label}': sum_len/len_query, f'range-{label}': list(temp_list_pd['range'])}])], ignore_index=True)

    return coverage_pd

def coverage_statistic(genus_name, acc_n, max_key, que):
    label_iden = {'original': 0, 'pident_90': 90, 'pident_95': 95}
    folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}/{acc_n}'
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if seq_record.id != max_key:
            os.chdir('/active-data/temp/blastn')
            query_file = open(f'temp_query-{acc_n}.fasta', 'w+')
            SeqIO.write(seq_record, query_file, "fasta")
            query_file.close()
            len_query = len(seq_record)
            os.system(f'blastn -query temp_query-{acc_n}.fasta -db /active-data/analysis_results/chr_pla/genus/blast_data/{genus_name}/nucleotide_seq.blastdb -out blastn_results-{acc_n}.txt -evalue 1e-50 -max_target_seqs 100000 -max_hsps 3000 -outfmt 6 -num_threads 1')
            head = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
            align_result = pd.read_csv(f'blastn_results-{acc_n}.txt', sep = '\t', engine = 'python', header = None, names = head)
            os.system(f'rm temp_query-{acc_n}.fasta')
            os.system(f'rm blastn_results-{acc_n}.txt')
            coverage_pd = pd.DataFrame()
            for label in label_iden:
                temp_result = align_result[(align_result['pident'] >= label_iden[label])].reset_index(drop=True)
                temp_pd = coverage_calcu(temp_result, label, len_query)
                if coverage_pd.empty:
                    coverage_pd = temp_pd.copy()
                else:
                    coverage_pd = pd.merge(coverage_pd, temp_pd, on='id', how='outer')
            os.chdir(folder)
            coverage_pd.to_csv(f'{seq_record.id}_coverage.csv', index=False)
    handle.close()
    que.put(1)

In [3]:
from tqdm import tqdm
import multiprocessing

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    
    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 32
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)

    NMS_count = 0
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        
        max_key, max_num = max_dict(chr_data | pla_data)
        pool.apply_async(coverage_statistic, (genus_name, acc_n, max_key, que))
    
    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:
                continue
    
    pool.join()

Escherichia: 100%|████████████████████████████████████████████| 4.20k/4.20k [9:46:48<00:00, 8.38s/B]
Klebsiella: 100%|████████████████████████████████████████████| 3.55k/3.55k [11:35:43<00:00, 11.7s/B]
Staphylococcus: 100%|███████████████████████████████████████████| 2.42k/2.42k [44:41<00:00, 1.11s/B]
Pseudomonas: 100%|██████████████████████████████████████████████| 2.34k/2.34k [17:38<00:00, 2.21B/s]
Bacillus: 100%|█████████████████████████████████████████████████| 1.98k/1.98k [29:36<00:00, 1.11B/s]
Salmonella: 100%|███████████████████████████████████████████████| 1.85k/1.85k [32:53<00:00, 1.07s/B]
Streptococcus: 100%|████████████████████████████████████████████| 1.60k/1.60k [01:21<00:00, 19.5B/s]
Streptomyces: 100%|█████████████████████████████████████████████| 1.36k/1.36k [48:03<00:00, 2.12s/B]
Acinetobacter: 100%|████████████████████████████████████████████| 1.23k/1.23k [25:04<00:00, 1.22s/B]
Helicobacter: 100%|█████████████████████████████████████████████████| 416/416 [00:15<00:00,